# ClaimSense AI - Fine-tuning with Unsloth

**Mistral AI Worldwide Hackathon - Track 1: Fine-tuning**

Fine-tunes Mistral-7B on 39,000+ insurance claims using Unsloth (5x faster training).

## Quick Start
1. **Enable GPU:** Runtime → Change runtime type → T4 GPU
2. **Run all cells** (Ctrl+F9)
3. Model auto-uploads to HuggingFace when done

Training data: https://huggingface.co/datasets/pramodmisra/claimsense-training-data

In [ ]:
# Install dependencies (takes ~2 min)
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets wandb huggingface_hub

In [ ]:
# Configuration - UPDATE THESE VALUES
HF_USERNAME = "pramodmisra"
MODEL_NAME = "claimsense-ai-v1"
DATASET_ID = "pramodmisra/claimsense-training-data"

# Login to HuggingFace (will prompt for token)
from huggingface_hub import login
import wandb

login()  # Enter your HF token when prompted
wandb.login()  # Enter your W&B key when prompted
print("Logged in!")

In [ ]:
# Load model with Unsloth (4-bit quantization for T4 GPU)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

print(f"Model loaded! GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Add LoRA adapters for efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("LoRA adapters added!")

In [ ]:
# Load training data from HuggingFace
from datasets import load_dataset

def format_prompts(examples):
    """Format messages into Mistral instruct format."""
    texts = []
    for messages in examples['messages']:
        text = ""
        for msg in messages:
            if msg['role'] == 'user':
                text += f"<s>[INST] {msg['content']} [/INST]"
            else:
                text += f" {msg['content']}</s>"
        texts.append(text)
    return {"text": texts}

# Load from HuggingFace
train_dataset = load_dataset(DATASET_ID, data_files="train.jsonl", split="train")
eval_dataset = load_dataset(DATASET_ID, data_files="eval.jsonl", split="train")

# Use subset for faster training (full dataset = 35k examples)
train_dataset = train_dataset.shuffle(seed=42).select(range(min(5000, len(train_dataset))))
eval_dataset = eval_dataset.shuffle(seed=42).select(range(min(500, len(eval_dataset))))

train_dataset = train_dataset.map(format_prompts, batched=True)
eval_dataset = eval_dataset.map(format_prompts, batched=True)

print(f"Training examples: {len(train_dataset)}")
print(f"Evaluation examples: {len(eval_dataset)}")

In [ ]:
# Configure training with W&B logging
from trl import SFTTrainer
from transformers import TrainingArguments

wandb.init(
    project="claimsense-ai",
    name="claimsense-mistral-v1",
    tags=["hackathon", "mistral", "insurance", "fine-tuning", "unsloth"]
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        output_dir="./claimsense-output",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=200,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        report_to=["wandb"],
    ),
)
print("Trainer configured!")

In [ ]:
# Start training! (takes ~1-2 hours on T4)
print("="*60)
print("Starting fine-tuning... This will take ~1-2 hours on T4 GPU")
print("="*60)

trainer_stats = trainer.train()

print("\n" + "="*60)
print("Training complete!")
print(f"Total training time: {trainer_stats.metrics['train_runtime']/60:.1f} minutes")
print("="*60)

In [ ]:
# Test the fine-tuned model
FastLanguageModel.for_inference(model)

test_prompt = """<s>[INST] Analyze this insurance claim for potential fraud:

Customer reports laptop stolen from unlocked car. Third claim this year for similar items.
No police report filed. Requesting full replacement value of $3,500. [/INST]"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("="*60)
print("TEST RESPONSE:")
print("="*60)
print(response.split("[/INST]")[-1].strip())

In [ ]:
# Save and upload to HuggingFace
print(f"Uploading model to: https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")

# Save locally
model.save_pretrained(MODEL_NAME)
tokenizer.save_pretrained(MODEL_NAME)

# Push to HuggingFace Hub
model.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}")
tokenizer.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}")

print(f"\nModel uploaded to: https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")

In [ ]:
# Finish W&B logging
wandb.finish()

print("\n" + "="*60)
print("ALL DONE!")
print("="*60)
print(f"\nModel: https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")
print(f"W&B Dashboard: https://wandb.ai/claimsense-ai")
print("\nNext steps:")
print("1. Deploy demo to HuggingFace Spaces")
print("2. Record video demo")
print("3. Submit to hackathon!")